# TUTORIAL: Next Sentence Prediction with BERT

- BOOK: [Dive Into DL](https://d2l.ai/chapter_natural-language-processing-pretraining/bert.html)
- NOTEBOOK: [Finetune BERT with IMDB](https://github.com/ufdatastudio/predictions/blob/development/notebook_experiments/tutorial-text_classification_hf-base.ipynb)
- NOTEBOOK: [Finetune BERT with TOLSA-M](https://github.com/ufdatastudio/predictions/blob/development/notebook_experiments/tutorial-text_classification_hf-tolsa_m.ipynb)

In [1]:
import torch
from torch import nn
from d2l import torch as d2l

## Input Representation

BERT's text input type is both single and multiple

1. Single Text Input $ \rightarrow $ Tasks with Examples
2. Multiple Text Input $ \rightarrow $ Tasks with Examples
3. CODE: Get Tokens and Segments
4. CODE: BERT Encoder with PyTorch

### Single Text Input Tasks $ \rightarrow $ Tasks with Examples

| # | Task | Input Example | Output Example |
|---|------|---------------|----------------|
| 1 | Sentiment Analysis | "I absolutely loved this movie." | Positive |
| 2 | Topic Classification | "The Federal Reserve announced a change in interest rates." | Finance |
| 3 | Text Tagging (Named Entity Recognition) | "Barack Obama was born in Hawaii." | Barack Obama → PERSON, Hawaii → LOCATION |
| 4 | Language Identification | "Bonjour tout le monde." | French |
| 5 | Spam Detection | "Congratulations! You have won a free iPhone." | Spam |
| 6 | Emotion Classification | "I am so excited for graduation!" | Joy |
| 7 | Toxicity Detection | "You are terrible at everything." | Toxic |

---

Single: Concatenation of the special classification token `<cls>`, tokens of a text sequence, and the special separation token `<sep>`.

<div align="center"><b>Task: 1 (SA)</b></div>

BERT Input Sequence 1:
    ```
    [CLS] I absolutely loved this movie . [SEP]
    ```

Segment IDs:
    ```
    0 0 0 0 0 0 0 0
    ```

BERT Input Sequence 2:
    ```
    [CLS] The film was a complete waste of time . [SEP]
    ```
    
Segment IDs:
    ```
    0 0 0 0 0 0 0 0 0 0
    ```

### Multiple Text Input Tasks $ \rightarrow $ Tasks with Examples

| # | Task | Input Example 1 | Input Example 2 | Output Example |
|---|------|-----------------|-----------------|----------------|
| 1 | Natural Language Inference (NLI) | Premise: "A man is playing guitar." | Hypothesis: "A person is making music." | Entailment |
| 2 | Semantic Textual Similarity | Sentence 1: "The cat sat on the mat." | Sentence 2: "A cat is sitting on a rug." | Similarity = High |
| 3 | Duplicate Question Detection | Question 1: "How do I learn Python?" | Question 2: "What's the best way to study Python?" | Duplicate |
| 4 | Paraphrase Identification | Sentence 1: "The meeting starts at 9." | Sentence 2: "The meeting begins at 9." | Paraphrase |
| 5 | Information Retrieval / Document Ranking | Query: "Symptoms of diabetes" | Document: "Common symptoms include thirst and fatigue." | Relevant |
| 6 | Question Answering | Context: "Paris is the capital of France." | Question: "What is the capital of France?" | Paris |
| 7 | Dialogue Response Selection | User Message: "What's the weather today?" | Candidate Response: "It will be sunny and 75°F." | Appropriate Response |

Multiple: Concatenation of `<cls>`, tokens of the first text sequence, `<sep>`, tokens of the second text sequence, and `<sep>`.

<div align="center"><b>Task: 1 (NLI) — Pair 1</b></div>

BERT Input Sequence Pair 1:
    ```
    [CLS] A man is playing guitar . [SEP] A person is making music . [SEP]
    ```

Segment IDs:
    ```
    0 0 0 0 0 0 0 0 1 1 1 1 1 1 1
    ```

BERT Input Sequence Pair 2:
    ```
    [CLS] A woman is running outside . [SEP] A person is exercising . [SEP]
    ```

Segment IDs:
    ```
    0 0 0 0 0 0 0 0 1 1 1 1 1 1
    ```

### CODE: Get Tokens and Segments

In [2]:
# @save
def get_tokens_and_segments(tokens_a, tokens_b=None):
    """Get tokens of the BERT input sequence and their segment IDs."""
    tokens = ['<cls>'] + tokens_a + ['<sep>']
    # 0 and 1 are marking segment A and B, respectively
    segments = [0] * (len(tokens_a) + 2)
    if tokens_b is not None:
        tokens += tokens_b + ['<sep>']
        segments += [1] * (len(tokens_b) + 1)
    return tokens, segments

#### Image of BERT across input + token embeddings + segment embeddings + positional embeddings

![alt text](<Screenshot 2026-09-19 at 13.14.50.png>)

### CODE: BERT Encoder + Transformer Encoder

> CODE: Transformer Block

- Becuase I'm getting `AttributeError: module 'd2l.torch' has no attribute 'TransformerEncoderBlock'`, so UF's claude-sonnect-4.6 generated the below.

In [5]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, num_hiddens, ffn_num_hiddens, num_heads, dropout, use_bias=False):
        super().__init__()
        self.attention = nn.MultiheadAttention(num_hiddens, num_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(num_hiddens, ffn_num_hiddens),
            nn.ReLU(),
            nn.Linear(ffn_num_hiddens, num_hiddens)
        )
        self.norm1 = nn.LayerNorm(num_hiddens)
        self.norm2 = nn.LayerNorm(num_hiddens)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X, valid_lens=None):
        attn_out, _ = self.attention(X, X, X)
        X = self.norm1(X + self.dropout(attn_out))
        ffn_out = self.ffn(X)
        return self.norm2(X + self.dropout(ffn_out))

# Monkey-patch it onto d2l
d2l.TransformerEncoderBlock = TransformerEncoderBlock

In [6]:
#@save
class BERTEncoder(nn.Module):
    """BERT encoder.
    
    Defines its architectual components in `__init__()` + stack of Tranform encoder blocks 
    Defines the foward method (shows how data flows)

    """
    def __init__(self, vocab_size, num_hiddens, ffn_num_hiddens, num_heads,
                 num_blks, dropout, max_len=1000, **kwargs):
        super(BERTEncoder, self).__init__(**kwargs)
        
        """ Embedding layers """
        self.token_embedding = nn.Embedding(vocab_size, num_hiddens) # word level (token) embedding
        self.segment_embedding = nn.Embedding(2, num_hiddens) # distinguishing text pairs

        """Transformer encoder blocks """
        self.blks = nn.Sequential()
        for i in range(num_blks):
            self.blks.add_module(f"{i}", d2l.TransformerEncoderBlock(
                num_hiddens, ffn_num_hiddens, num_heads, dropout, True))

        """ Learnable embedding layer """
        # In BERT, positional embeddings are learnable, thus we create a
        # parameter of positional embeddings that are long enough
        self.pos_embedding = nn.Parameter(torch.randn(1, max_len,
                                                      num_hiddens))

    def forward(self, tokens, segments, valid_lens):
        # Shape of `X` remains unchanged in the following code snippet:
        # (batch size, max sequence length, `num_hiddens`)
        X = self.token_embedding(tokens) + self.segment_embedding(segments)
        X = X + self.pos_embedding[:, :X.shape[1], :]
        for blk in self.blks:
            X = blk(X, valid_lens)
        return X

> Create instance and initialize its parameters

In [ ]:
vocab_size, num_hiddens, ffn_num_hiddens, num_heads = 10000, 768, 1024, 4
ffn_num_input, num_blks, dropout = 768, 2, 0.2
encoder = BERTEncoder(
    vocab_size, 
    num_hiddens, 
    ffn_num_hiddens, 
    num_heads,
    num_blks, dropout
    )